## Criteria evaluator

### Instalacja bibliotek

In [1]:
! pip install --upgrade langsmith langchain langchain-core langchain_openai langchain_classic langchain-text-splitters

In [2]:
from langsmith import traceable
from langchain_classic.evaluation import load_evaluator, Criteria
from langsmith import Client
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import json
import uuid
import os

load_dotenv()

True

### Import bibliotek i konfiguracja LangSmith

In [3]:
# Włącz śledzenie LangSmith (wymaga konta LangSmith):
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "kurs-demo"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGSMITH_API_KEY"] = "<TWÓJ_KLUCZ>" #załączony w .env

### Wygenerowanie odpowiedzi LLM

In [4]:
def message_to_text(message) -> str:
    """Zamienia AIMessage / dict / str na zwykły tekst możliwy do zapisania w LangSmith."""
    content = getattr(message, "content", message)

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        chunks = []
        for item in content:
            if isinstance(item, dict):
                chunks.append(str(item.get("text") or item.get("content") or item))
            else:
                chunks.append(str(item))
        return "\n".join(chunks)

    if isinstance(content, dict):
        return json.dumps(content, ensure_ascii=False)

    return str(content)

In [5]:
client = Client()

dataset_inputs = [
    "Why people don't have 3 legs?",
    "Why people are not flying?",
]

llm_test = ChatOpenAI(model="gpt-5.4", temperature=0.1, max_tokens=2048)
llm_gen = ChatOpenAI(model="gpt-5-mini", temperature=0.1, max_tokens=2048)

dataset_outputs = [
    {"answer": message_to_text(llm_test.invoke(question))}
    for question in dataset_inputs
]

print(json.dumps(dataset_outputs, indent=2, ensure_ascii=False))


[
  {
    "answer": "Humans don’t have 3 legs mainly because of **evolution and body design**.\n\n### Short answer:\nOur bodies evolved from ancestors with **bilateral symmetry** — basically, a left side and a right side. That body plan naturally supports **two legs**, two arms, two eyes, etc.\n\n### Why not 3 legs?\n1. **Evolution works from existing designs**  \n   It doesn’t build creatures from scratch. Early vertebrates already had a symmetrical 4-limb body plan, and humans inherited that pattern.\n\n2. **Two legs are enough for walking**  \n   Humans evolved **bipedalism**, meaning walking on two legs. This freed our hands for carrying tools, climbing, and other tasks.\n\n3. **Three legs would complicate development**  \n   Growing an extra limb would require major changes in:\n   - skeleton\n   - muscles\n   - nerves\n   - balance system\n   - brain coordination\n\n4. **Symmetry is efficient**  \n   Left-right symmetry helps with movement and balance. A third leg in the middle w

In [6]:
eval_llm = ChatOpenAI(model="gpt-5.4", temperature=0, max_tokens=2048)

def make_langchain_string_evaluator(key: str, evaluator):
    """
    Adapter dla ewaluatorów LangChain Classic.
    """
    def _evaluate(inputs: dict, outputs: dict, reference_outputs: dict | None = None) -> dict:
        kwargs = {
            "prediction": message_to_text(outputs.get("answer", outputs)),
            "input": message_to_text(inputs.get("question", inputs)),
        }

        if reference_outputs:
            kwargs["reference"] = message_to_text(reference_outputs.get("answer", reference_outputs))

        result = evaluator.evaluate_strings(**kwargs)

        return {
            "key": key,
            "score": result.get("score"),
            "value": result.get("value"),
            "comment": result.get("reasoning") or result.get("comment") or "",
        }

    _evaluate.__name__ = key
    return _evaluate

correctness_evaluator = make_langchain_string_evaluator(
    "correctness", load_evaluator("labeled_criteria", llm=eval_llm, criteria="correctness"),
)

criteria_evaluators = [
    make_langchain_string_evaluator("helpfulness",load_evaluator("criteria", llm=eval_llm, criteria=Criteria.HELPFULNESS)),
    make_langchain_string_evaluator("relevance", load_evaluator("criteria", llm=eval_llm, criteria=Criteria.RELEVANCE)),
    make_langchain_string_evaluator("coherence",load_evaluator("criteria", llm=eval_llm, criteria=Criteria.COHERENCE)),
    make_langchain_string_evaluator("conciseness", load_evaluator("criteria", llm=eval_llm, criteria=Criteria.CONCISENESS)),
    make_langchain_string_evaluator("harmfulness", load_evaluator("criteria", llm=eval_llm, criteria=Criteria.HARMFULNESS)),
    make_langchain_string_evaluator("maliciousness", load_evaluator("criteria", llm=eval_llm, criteria=Criteria.MALICIOUSNESS)),
]

evaluators = [
    correctness_evaluator,
    *criteria_evaluators,
]


### Uruchomienie i integracja z LangSmith

In [7]:
dataset_name = f"existential questions run:{uuid.uuid4()}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="evaluate LLM output",
)

examples = [
    {
        "inputs": {"question": question},
        "outputs": reference_output,
    }
    for question, reference_output in zip(dataset_inputs, dataset_outputs)
]

client.create_examples(dataset_id=dataset.id, examples=examples)

print(f"Created dataset: {dataset_name}")


Created dataset: existential questions run:6a80ab60-1ce8-4a0b-ab29-a1c0c38eaa8a


In [8]:
@traceable(name="generate_answer")
def generate_answer(inputs: dict) -> dict:
    """Funkcja / chain oceniany przez LangSmith."""
    response = llm_gen.invoke(inputs["question"])
    return {"answer": message_to_text(response)}

results = client.evaluate(
    generate_answer,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix=dataset_name,
    max_concurrency=2,
)

print(results)


View the evaluation results for experiment: 'existential questions run:6a80ab60-1ce8-4a0b-ab29-a1c0c38eaa8a-694552e7' at:
https://smith.langchain.com/o/3e1f981e-76ef-5491-9a42-e33f3bdfeba4/datasets/3399528a-ac7e-4252-969c-90ffc866e84d/compare?selectedSessions=3ea4bda3-60e0-4e00-a286-ad0017675ec8




0it [00:00, ?it/s]

/tmp/ipykernel_36508/3037230694.py:16: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use references, use the labeled_criteria instead.
  result = evaluator.evaluate_strings(**kwargs)
/tmp/ipykernel_36508/3037230694.py:16: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use references, use the labeled_criteria instead.
  result = evaluator.evaluate_strings(**kwargs)
/tmp/ipykernel_36508/3037230694.py:16: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use references, use the labeled_criteria instead.
  result = evaluator.evaluate_strings(**kwargs)
/tmp/ipykernel_36508/3037230694.py:16: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use references, use the labeled_criteria instead.
  result = evaluator.evaluate_strings(**kwargs)
/tmp/ipykernel_36508/3037230694.py:16: UserWarning: Ignoring reference in CriteriaEvalChain, as it is not expected.
To use reference

<ExperimentResults existential questions run:6a80ab60-1ce8-4a0b-ab29-a1c0c38eaa8a-694552e7>
